# 第4章 · 数据清洗与输入输出

本单元可以单独打开并从头运行，不依赖其他 Notebook 的变量或输出文件。全部小数据为本课程自编合成数据，不代表真实学生、订单或股票行情。

**学习方法**：先说明每行含义与预期结果，再运行代码；练习答案位于折叠单元格。使用课程 `.venv`，无需下载数据或额外安装依赖。

**阅读参考**：[Python for Data Analysis 对应章节](https://wesmckinney.com/book/data-cleaning)。本单元文字、数据与练习为课程自行编写。

### 1. 原始登记表：保留文本才能看见问题

教学合成数据，每行一次成绩登记。故意保留首尾空格、重复记录、缺考和不合理成绩；学号按字符串读入以保留前导零。

In [1]:
from io import StringIO
import pandas as pd
raw_csv = """学号,姓名,班级,成绩,日期
001, 小林 ,A,82,2026-09-01
002,小周,a,95,2026-09-01
003,小陈,B,缺考,2026-09-02
004,小许,B,108,2026-09-02
001, 小林 ,A,82,2026-09-01
005,小吴, B ,76,日期待核"""
raw = pd.read_csv(StringIO(raw_csv), dtype="string")
raw

,学号,姓名,班级,成绩,日期
0,001,小林,A,82,2026-09-01
1,002,小周,a,95,2026-09-01
2,003,小陈,B,缺考,2026-09-02
3,004,小许,B,108,2026-09-02
4,001,小林,A,82,2026-09-01
5,005,小吴,B,76,日期待核


### 2. 先检查，再清洗

info 看类型和非缺失数，duplicated 找整行重复。这里“缺考”是普通文本，尚不会被 isna 自动识别。

In [2]:
raw.info()
print(raw.isna().sum())
print("整行重复数：", raw.duplicated().sum())
raw.loc[raw.duplicated(keep=False)]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   学号      6 non-null      string
 1   姓名      6 non-null      string
 2   班级      6 non-null      string
 3   成绩      6 non-null      string
 4   日期      6 non-null      string
dtypes: string(5)
memory usage: 372.0 bytes
学号    0
姓名    0
班级    0
成绩    0
日期    0
dtype: int64
整行重复数： 1


,学号,姓名,班级,成绩,日期
0,001,小林,A,82,2026-09-01
4,001,小林,A,82,2026-09-01


### 3. str 方法：把单个字符串操作扩展到整列

copy 保留原始登记。统一班级格式后再分组，否则 A 和 a 会被算成两个班。

In [3]:
clean = raw.copy()
clean["姓名"] = clean["姓名"].str.strip()
clean["班级"] = clean["班级"].str.strip().str.upper()
print(clean["班级"].value_counts())
print(clean["姓名"].str.contains("小", na=False))

班级
A    3
B    3
Name: count, dtype: Int64
0    True
1    True
2    True
3    True
4    True
5    True
Name: 姓名, dtype: boolean


### 4. 类型转换：转换失败的行必须可追踪

先把明确的缺考编码改成缺失，再转数值。coerce 将无法解析的值变成缺失；保留原列才能区分缺考与格式错误。

In [4]:
score_text = clean["成绩"].replace("缺考", pd.NA)
clean["分数"] = pd.to_numeric(score_text, errors="coerce")
clean["登记日"] = pd.to_datetime(clean["日期"], errors="coerce")
parse_failed = score_text.notna() & clean["分数"].isna()
print(clean.loc[parse_failed | clean["登记日"].isna()])
print(clean.dtypes)

    学号  姓名 班级  成绩    日期  分数 登记日
5  005  小吴  B  76  日期待核  76 NaT
学号     string[python]
姓名     string[python]
班级     string[python]
成绩     string[python]
日期     string[python]
分数              Int64
登记日    datetime64[ns]
dtype: object


### 5. 重复键不等于重复行

本例每名学生只有一次考试，学号应唯一。已确认整行相同的重复登记可以删除；若同一学号分数不同，应先核实，不能任意保留。

In [5]:
print(clean.loc[clean.duplicated("学号", keep=False)])
clean = clean.drop_duplicates().reset_index(drop=True)
print("清理后记录数：", len(clean))
print("学号唯一：", clean["学号"].is_unique)

    学号  姓名 班级  成绩          日期  分数        登记日
0  001  小林  A  82  2026-09-01  82 2026-09-01
4  001  小林  A  82  2026-09-01  82 2026-09-01
清理后记录数： 5
学号唯一： True


### 6. 异常值与缺失值分开记录

考试满分 100，因此 108 是待核异常，不能直接截成 100。缺考也不等于 0 分。设置标记后，在统计列中排除异常。

In [6]:
clean["成绩异常"] = (clean["分数"].notna() &
                       ~clean["分数"].between(0, 100))
clean["有效成绩"] = clean["分数"].mask(clean["成绩异常"])
print(clean[["学号", "成绩", "成绩异常", "有效成绩"]])
print("有效成绩均值：", clean["有效成绩"].mean())

    学号   成绩   成绩异常  有效成绩
0  001   82  False    82
1  002   95  False    95
2  003   缺考  False  <NA>
3  004  108   True  <NA>
4  005   76  False    76
有效成绩均值： 84.33333333333333


**先动手**：计算有效成绩数量、缺考数量、异常成绩数量。为什么不能把三者混成一类？

In [7]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
print(clean["有效成绩"].count())       # 3
print(clean["成绩"].eq("缺考").sum())  # 1
print(clean["成绩异常"].sum())         # 1
# 缺考是未观测，108是录入错误候选，处理依据不同。
```
</details>

### 7. 删除与填充：输出改变了什么

比较同一列的有效值平均与填零平均。dropna 的 subset 明确本次分析依赖哪些列；不能因为日期缺失就删除不依赖日期的成绩。

In [8]:
print(clean["有效成绩"].mean())
print(clean["有效成绩"].fillna(0).mean())
usable_scores = clean.dropna(subset=["有效成绩"])
print(usable_scores[["学号", "有效成绩"]])

84.33333333333333
50.6
    学号  有效成绩
0  001    82
1  002    95
4  005    76


### 8. 分类与映射：明确区间边界

cut 按固定阈值分段，right=False 表示左闭右开。100 包含在 [90,101) 内；没有有效成绩的行仍保留缺失等级。

In [9]:
clean["等级"] = pd.cut(clean["有效成绩"],
    bins=[0, 60, 80, 90, 101], right=False,
    labels=["待提高", "合格", "良好", "优秀"])
clean["班级名称"] = clean["班级"].map({"A": "一班", "B": "二班"})
clean[["学号", "有效成绩", "等级", "班级名称"]]

,学号,有效成绩,等级,班级名称
0,001,82,良好,一班
1,002,95,优秀,一班
2,003,<NA>,NaN,二班
3,004,<NA>,NaN,二班
4,005,76,合格,二班


### 9. 导出与读回：CSV 不保存所有类型

先在内存中演示 CSV 往返。读回时显式指定学号类型；分类类型和日期类型需要重建。真实文件路径可替换 StringIO。

In [10]:
csv_text = clean.to_csv(index=False)
restored = pd.read_csv(StringIO(csv_text), dtype={"学号": "string"})
print(restored["学号"].tolist())
print(len(raw), len(clean), len(restored))
assert len(raw) == 6 and len(clean) == 5
assert restored["学号"].iloc[0] == "001"

['001', '002', '003', '004', '005']
6 5 5


### 10. 本单元练习：给出可解释的清洗记录

交付三项结果：保留几行、排除哪些成绩、哪些日期待核。下一单元学习把这类记录与另一张信息表合并。

In [11]:
audit = pd.Series({
    "输入行数": len(raw), "保留行数": len(clean),
    "有效成绩数": clean["有效成绩"].count(),
    "异常成绩数": clean["成绩异常"].sum(),
    "待核日期数": clean["登记日"].isna().sum()})
print(audit)

输入行数     6
保留行数     5
有效成绩数    3
异常成绩数    1
待核日期数    1
dtype: int64


**先动手**：找出有效成绩低于 80 的学生，保留学号、姓名、班级、有效成绩。解释为什么没有把缺考者选进来。

In [12]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
clean.loc[clean["有效成绩"].lt(80).fillna(False),
          ["学号", "姓名", "班级", "有效成绩"]]
# 小吴，76分；缺失成绩不能据此判断是否低于80。
```
</details>